In [1]:
from gliclass.data_processing import GLiClassDataset
from transformers import AutoModel, AutoConfig
from gliclass.config import GLiClassModelConfig
from gliclass.model import GLiClassModel, GLiClassBiEncoder, GLiClassAudio
from transformers import AutoConfig, AutoTokenizer, AutoFeatureExtractor
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import torchaudio, torch

/home/werent4/GLiClass/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
encoder_config = AutoConfig.from_pretrained("facebook/wav2vec2-base-960h")
audiocfg = AutoConfig.from_pretrained("facebook/wav2vec2-base-960h")
audio_feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")

In [3]:
glicalss_config = GLiClassModelConfig(
    encoder_config=encoder_config,
    encoder_model="answerdotai/ModernBERT-base",
    audio_model_name="facebook/wav2vec2-base-960h",
    audio_model_config=audiocfg,
    class_token_index=len(tokenizer),
    text_token_index=len(tokenizer)+1,
    audio_token_index=len(tokenizer)+2, 
    pooling_strategy="first",
    scorer_type="audio-token-dot",
    use_lstm=False,
    focal_loss_alpha=-1,
    focal_loss_gamma=-1,
    contrastive_loss_coef=0.0,
    normalize_features=False,
    extract_text_features=False,
    architecture_type='audio-bi-encoder',
    prompt_first=True,
    squeeze_layers=False,
    shuffle_labels=True
)

model = GLiClassModel(glicalss_config, from_pretrained=True)
new_words = ["<<LABEL>>", "<<SEP>>", "<<AUDIO>>"]
tokenizer.add_tokens(new_words, special_tokens=True)
model.resize_token_embeddings(len(tokenizer), None)

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(50371, 768, padding_idx=50283)

In [4]:
import json
data = json.load(open("./datasets/processed_dataset.json", "r"))

In [5]:
train_dataset = GLiClassDataset(
    data,
    tokenizer,
    1024,
    "multi_label_classification",
    'audio-bi-encoder',
    True,
    labels_tokenizer=tokenizer,
    audio_features_extractor= audio_feature_extractor,
    sampling_rate= 16000,
    max_duration_s=20
)

Total labels:  7
Audio parameters: sampling_rate=16000, max_duration=20s, max_samples=320000


In [6]:
exmpl = train_dataset[0]

In [7]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/tmp/ipykernel_1997/2331616996.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)
/tmp/ipykernel_1997/2331616996.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)
/tmp/ipykernel_1997/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [11]:
exmpl["audio_input"].unsqueeze(0)  

tensor([[0.0410, 0.0344, 0.0331,  ..., 0.0002, 0.0002, 0.0002]])

In [12]:
model(input_ids, attention_mask, exmpl["audio_input"].unsqueeze(0), labels=labels, return_dict=True)

GLiClassOutput(loss=tensor(1.9286, grad_fn=<NegBackward0>), logits=tensor([[0.4004, 0.4535, 0.3614, 0.4315, 0.4520, 0.4428, 0.4340]],
       grad_fn=<MulBackward0>), hidden_states=None, attentions=None, text_embeddings=None, class_embeddings=None)

In [8]:
print("input_ids type:", type(exmpl['input_ids']))
print("attention_mask type:", type(exmpl['attention_mask']))
print("labels type:", type(exmpl['labels']))

input_ids type: <class 'list'>
attention_mask type: <class 'list'>
labels type: <class 'torch.Tensor'>


In [9]:
input_ids = torch.tensor(exmpl['input_ids']).unsqueeze(0)  
attention_mask = torch.tensor(exmpl['attention_mask']).unsqueeze(0)  
labels = torch.tensor(exmpl['labels']).unsqueeze(0)  

/var/tmp/ipykernel_29360/2331616996.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(exmpl['labels']).unsqueeze(0)


In [10]:
exmpl['labels']

tensor([1., 0., 0., 0., 0., 0., 0.])

In [11]:
print("Labels shape:", labels.shape)
print("Labels:", labels)
print("Problem type:", model.config.problem_type)

Labels shape: torch.Size([1, 7])
Labels: tensor([[1., 0., 0., 0., 0., 0., 0.]])
Problem type: None


In [12]:
exmpl['audio_input'].unsqueeze(0).shape

torch.Size([1, 80000])

In [14]:
model(input_ids, attention_mask, exmpl["audio_input"], labels=labels, return_dict=True)

GLiClassOutput(loss=tensor(4.1938, grad_fn=<NegBackward0>), logits=tensor([[-0.5545,  3.1775,  0.8603,  0.8491,  1.9720, -0.0429, -0.4186]],
       grad_fn=<MulBackward0>), hidden_states=None, attentions=None, text_embeddings=None, class_embeddings=None)